# 6. Воспроизводимый отчёт

**Цель:** выполнить поддерживаемый `GraphEVTPipeline` и сохранить численный результат, manifest и краткий отчёт. Ни LLM, ни вымышленные поля результата для этого не требуются.

In [1]:
from pathlib import Path
from importlib.metadata import version
from hashlib import sha256
import json
import numpy as np
import pandas as pd
from graph_evt_agent import EVTConfig, GraphConfig, GraphEVTPipeline, InputConfig

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/generated/fractal_extreme_series.csv").exists())
DATA = ROOT / "data/generated/fractal_extreme_series.csv"
SEED = 42
df = pd.read_csv(DATA)
x = df["value"].to_numpy()
assert np.isfinite(x).all() and x.ndim == 1

In [2]:
BASELINE_SIZE = int(0.75 * len(x))
parameters = {
    "baseline_size": BASELINE_SIZE,
    "tail_quantile": 0.95,
    "alarm_probability": 0.999,
    "persistence": 3,
}
evt = EVTConfig(**parameters)
result = GraphEVTPipeline(
    evt,
    graph=GraphConfig(random_state=SEED),
    input_config=InputConfig(missing="error"),
    verbose=True,
).run_detailed(x)
detection = result.detection

artifact_dir = ROOT / "results/1-d/06_report"
artifact_dir.mkdir(parents=True, exist_ok=True)
indicator_path = artifact_dir / "indicator.csv"
pd.DataFrame({
    "time": df["timestamp"],
    "value": x,
    "indicator": detection.indicator,
    "above_alarm_threshold": detection.indicator > detection.alarm_threshold,
    "is_baseline": np.arange(len(x)) < BASELINE_SIZE,
}).to_csv(indicator_path, index=False)

Graph-EVT pipeline:   0%|          | 0/3 [00:00<?, ?it/s]

In [3]:
manifest = {
    "data_file": str(DATA.relative_to(ROOT)),
    "data_sha256": sha256(DATA.read_bytes()).hexdigest(),
    "library": "graph-evt-agent",
    "library_version": version("graph-evt-agent"),
    "seed": SEED,
    "parameters": parameters,
    "input_profile": {
        "kind": result.input_profile.kind,
        "n_time": result.input_profile.n_time,
        "n_channels": result.input_profile.n_channels,
    },
    "detection": {
        "detected": detection.detected,
        "time_index": detection.time_index,
        "alarm_threshold": float(detection.alarm_threshold),
    },
    "artifacts": {"indicator_csv": indicator_path.name},
    "limitations": [
        "synthetic unlabelled one-dimensional series",
        "two-sided deviations are combined in one indicator",
        "no graph or source ranking is available for one channel",
        "results depend on baseline and EVT configuration",
    ],
}
with (artifact_dir / "manifest.json").open("w", encoding="utf-8") as file:
    json.dump(manifest, file, ensure_ascii=False, indent=2)

status = "обнаружено" if detection.detected else "не обнаружено"
summary = f"""# Краткий отчёт

На {len(x)} наблюдениях устойчивое превышение {status}.
Alarm-порог: {detection.alarm_threshold:.6g}; первый индекс: {detection.time_index}.
Численный ряд indicator сохранён в `indicator.csv`, параметры и ограничения — в `manifest.json`.
"""
(artifact_dir / "report.md").write_text(summary, encoding="utf-8")
manifest

{'data_file': 'data/generated/fractal_extreme_series.csv',
 'data_sha256': '8d13d380c47327db1ca8a5324fb73dc2d7106dee511d27c8543267104f6204f3',
 'library': 'graph-evt-agent',
 'library_version': '0.1.0',
 'seed': 42,
 'parameters': {'baseline_size': 1125,
  'tail_quantile': 0.95,
  'alarm_probability': 0.999,
  'persistence': 3},
 'input_profile': {'kind': 'univariate', 'n_time': 1500, 'n_channels': 1},
 'detection': {'detected': False,
  'time_index': None,
  'alarm_threshold': 9.200682122203109},
 'artifacts': {'indicator_csv': 'indicator.csv'},
 'limitations': ['synthetic unlabelled one-dimensional series',
  'two-sided deviations are combined in one indicator',
  'no graph or source ranking is available for one channel',
  'results depend on baseline and EVT configuration']}

In [4]:
required = {"data_sha256", "library_version", "parameters", "input_profile", "detection", "artifacts", "limitations"}
missing = required - set(manifest)
assert not missing, f"Неполный manifest: {sorted(missing)}"
assert len(manifest["data_sha256"]) == 64
summary

'# Краткий отчёт\n\nНа 1500 наблюдениях устойчивое превышение не обнаружено.\nAlarm-порог: 9.20068; первый индекс: None.\nЧисленный ряд indicator сохранён в `indicator.csv`, параметры и ограничения — в `manifest.json`.\n'

Повторный запуск при той же версии библиотеки и тех же входных байтах должен воспроизводить численный результат. Manifest фиксирует фактические поля публичного API, а не конфигурацию отсутствующего LLM.

**Самопроверка:** зачем хранить hash данных? Почему одной версии ноутбука недостаточно? Какие выводы нельзя переносить за пределы этого ряда?